# Notebook 2 — Teste de conexão com o Postgres

Este notebook testa a conectividade com o banco de dados PostgreSQL e executa a consulta MIMIC FHIR especificada.

In [1]:
import os
print(f"Diretório de trabalho: {os.getcwd()}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH', 'não definido')}")

Diretório de trabalho: /home/jovyan/work/notebooks
PYTHONPATH: /home/jovyan/work/src:


In [2]:
from nl2sql2nl.database import criar_conexao, executar_consulta

print("Módulos importados com sucesso!")

/opt/conda/lib/python3.11/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


2026-05-26 20:16:18 | INFO | nl2sql2nl.config | Configurações carregadas com sucesso.
Módulos importados com sucesso!


## Testando a conexão

In [3]:
# Tenta criar uma conexão com o banco de dados
try:
    conexao = criar_conexao()
    print("✓ Conexão bem-sucedida!")
    print(f"Status: {conexao.status}")
    conexao.close()
    print("✓ Conexão fechada com sucesso.")
except Exception as e:
    print(f"✗ Erro ao conectar: {e}")
    print(f"Tipo do erro: {type(e).__name__}")

2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Conectando ao Postgres em host.docker.internal:5432/mimic_fhir
2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Conexão estabelecida com sucesso.
✓ Conexão bem-sucedida!
Status: 1
✓ Conexão fechada com sucesso.


## Executando consulta MIMIC FHIR

Esta consulta retorna dados de pacientes, encontros e localizações no banco MIMIC FHIR.
Query original especificada no projeto.

In [4]:
# Consulta MIMIC FHIR
sql = """
select p.nome_familia, e.periodo_inicio, e.periodo_fim,
       el.periodo_inicio, el.periodo_fim, l.nome
from pacientes p
inner join encontros e on p.id = e.paciente_id
inner join encontros_localizacoes el on el.encontro_id = e.id
inner join localizacoes l on el.localizacao_id = l.id
where p.identificador = '10000032'
"""

print("Executando consulta...\n")

try:
    resultados = executar_consulta(sql)
    print(f"✓ Consulta executada com sucesso!")
    print(f"✓ Resultados encontrados: {len(resultados)} linhas")
except Exception as e:
    print(f"✗ Erro ao executar consulta: {e}")
    print(f"Tipo do erro: {type(e).__name__}")
    resultados = []

Executando consulta...

2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Conectando ao Postgres em host.docker.internal:5432/mimic_fhir
2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Conexão estabelecida com sucesso.
2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Executando consulta SQL.
2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Consulta retornou 9 linhas.
2026-05-26 20:16:18 | INFO | nl2sql2nl.database | Conexão encerrada.
✓ Consulta executada com sucesso!
✓ Resultados encontrados: 9 linhas


## Exibindo resultados como tabela

In [5]:
import pandas as pd

if resultados:
    # Converte para DataFrame do Pandas para exibição formatada
    df = pd.DataFrame(resultados)
    print(f"\n{len(df)} linhas de dados:\n")
    display(df)
else:
    print("Nenhum resultado retornado pela consulta.")


9 linhas de dados:



,nome_familia,periodo_inicio,periodo_fim,nome
0,Patient_10000032,2180-05-06 19:17:00,2180-05-06 23:30:00,Emergency Department
1,Patient_10000032,2180-05-06 23:30:00,2180-05-07 17:21:27,Transplant
2,Patient_10000032,2180-06-26 15:54:00,2180-06-26 21:31:00,Emergency Department
3,Patient_10000032,2180-06-26 21:31:00,2180-06-27 18:49:12,Transplant
4,Patient_10000032,2180-08-05 20:58:00,2180-08-06 01:44:00,Emergency Department
5,Patient_10000032,2180-08-06 01:44:00,2180-08-07 17:50:44,Transplant
6,Patient_10000032,2180-07-22 16:24:00,2180-07-23 05:54:00,Emergency Department
7,Patient_10000032,2180-07-23 14:00:00,2180-07-23 23:50:47,Medical Intensive Care Unit (MICU)
8,Patient_10000032,2180-07-23 23:50:47,2180-07-24 19:52:58,Transplant


## Resumo

Se você chegou aqui sem erros:
- ✓ Variáveis de ambiente (.env) foram carregadas corretamente
- ✓ Conexão com Postgres foi estabelecida
- ✓ Consulta SQL foi executada
- ✓ Dados foram retornados e formatados